# SeaDronesSee — ConvNeXt PT2E Graph Smoke Test (Colab)

Notebook này chỉ để kiểm tra nhánh **PT2E graph** có chạy đúng kiến trúc hay chưa trên Colab. Nó không train dài và không tối ưu kết quả. Luồng test gồm:

1. clone / pull repo
2. cài stack PT2E phù hợp
3. kiểm tra `torchvision nms` và detection import
4. tải dataset SeaDronesSee vào SSD tạm của Colab
5. tạo config runtime cho Colab
6. chạy `prepare_pt2e_backbone_qat`
7. chạy `convert_pt2e_backbone`
8. save / load lại INT8 artifact

Chọn **Runtime -> Change runtime type -> GPU** trước khi chạy.

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/EchteAI')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repository:', REPO)

## 1. Cài môi trường PT2E

Cell này gỡ stack torch cũ của Colab rồi cài stack PT2E tối thiểu. Sau khi chạy xong, **Runtime -> Restart session** rồi chạy lại notebook từ cell kế tiếp.

In [ ]:
def run(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio', 'torchao'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.12.1', 'torchvision==0.27.1'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'torchao==0.17.0'])
run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-e', '.[coco,pt2e]'])
print('Install done. Restart session now, then continue from the next cell.')

In [ ]:
import torch
import torchvision
from torchvision.ops import nms
from torchvision.models.detection import FasterRCNN

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
import torchao
print('torchao:', torchao.__version__)
print('torchvision nms import OK')
print('torchvision detection import OK')

## 2. Tải SeaDronesSee vào SSD tạm của Colab

Notebook dùng cùng nguồn WebDAV như notebook QAT cũ. Không đặt dataset trên Drive vì đọc ảnh qua Drive chậm.

In [ ]:
!apt-get -qq update && apt-get -qq install -y rclone

In [ ]:
DATA_ROOT = Path('/content/seadronessee')
DAV_URL = 'https://cloud.cs.uni-tuebingen.de/public.php/dav/files/ZZxX65FGnQ8zjBP/'
subprocess.run(['rclone', 'config', 'delete', 'seadrones'], check=False, capture_output=True)
subprocess.run(['rclone', 'config', 'create', 'seadrones', 'webdav', 'url', DAV_URL, 'vendor', 'nextcloud'], check=True)
subprocess.run([
    'rclone', 'copy', 'seadrones:Compressed Version', str(DATA_ROOT),
    '--progress', '--transfers', '8', '--checkers', '16', '--retries', '10',
], check=True)
required = [
    DATA_ROOT/'images/train',
    DATA_ROOT/'images/val',
    DATA_ROOT/'annotations/instances_train.json',
    DATA_ROOT/'annotations/instances_val.json',
]
assert all(path.exists() for path in required), [str(path) for path in required if not path.exists()]
print('Dataset ready at', DATA_ROOT)

## 3. Tạo runtime config cho Colab

In [ ]:
import yaml

base = yaml.safe_load((REPO / 'configs' / 'seadronessee_colab.yaml').read_text())
base['dataset']['train_images'] = str(DATA_ROOT / 'images/train')
base['dataset']['train_annotations'] = str(DATA_ROOT / 'annotations/instances_train.json')
base['dataset']['val_images'] = str(DATA_ROOT / 'images/val')
base['dataset']['val_annotations'] = str(DATA_ROOT / 'annotations/instances_val.json')
base['dataset']['test_images'] = str(DATA_ROOT / 'images/val')
base['dataset']['test_annotations'] = str(DATA_ROOT / 'annotations/instances_val.json')
base['quantization']['pt2e']['region'] = 'backbone'
base['quantization']['pt2e']['observer_warmup_epochs'] = 1
base['quantization']['pt2e']['observer_freeze_epochs'] = 1
base['quantization']['pt2e']['example_batch_size'] = 2
base['quantization']['pt2e']['maximum_batch_size'] = 2
base['quantization']['pt2e']['example_height'] = 256
base['quantization']['pt2e']['example_width'] = 320
RUNTIME_CONFIG = REPO / 'runtime_pt2e_smoke_colab.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding='utf-8')
print('Saved runtime config:', RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text()[:1400])

## 4. PT2E prepare smoke test

Pass condition:

- `prepare_pt2e_backbone_qat()` không lỗi
- `fake_quantizers > 0`
- phase switching chạy được

In [ ]:
from pipelines.convnext_qat.config import load_config
from pipelines.convnext_qat.models import build_fasterrcnn_convnext
from pipelines.convnext_qat.quantization.pt2e_qat import prepare_pt2e_backbone_qat, set_pt2e_qat_phase

config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
model = build_fasterrcnn_convnext(config)
model = prepare_pt2e_backbone_qat(model, config)
count = set_pt2e_qat_phase(model, 'observer_warmup')
print('fake_quantizers:', count)
count = set_pt2e_qat_phase(model, 'full')
print('fake_quantizers full:', count)
count = set_pt2e_qat_phase(model, 'frozen')
print('fake_quantizers frozen:', count)
print('PT2E prepare smoke test passed')

## 5. PT2E convert / artifact smoke test

Pass condition:

- prepared model forward được 1 batch
- convert sang INT8 được
- save / load artifact được
- cả prepared / converted / reloaded đều trả về detection output

In [ ]:
import torch
from pipelines.convnext_qat.data import build_coco_loader
from pipelines.convnext_qat.quantization.pt2e_qat import convert_pt2e_backbone, save_pt2e_int8_artifact, load_pt2e_int8_artifact

ARTIFACT_PATH = REPO / 'pt2e_smoke_int8.pt'
loader = build_coco_loader(config, 'test', shuffle=False, limit=1, batch_size=1)
images, targets = next(iter(loader))

model = model.cpu().eval()
with torch.inference_mode():
    prepared_outputs = model([img.cpu() for img in images])
print('Prepared keys:', prepared_outputs[0].keys())
print('Prepared boxes:', len(prepared_outputs[0]['boxes']))
if len(prepared_outputs[0]['scores']) > 0:
    print('Prepared top score:', float(prepared_outputs[0]['scores'][0]))

int8_model = convert_pt2e_backbone(model, inplace=False, compile_region=False).cpu().eval()
with torch.inference_mode():
    int8_outputs = int8_model([img.cpu() for img in images])
print('INT8 keys:', int8_outputs[0].keys())
print('INT8 boxes:', len(int8_outputs[0]['boxes']))
if len(int8_outputs[0]['scores']) > 0:
    print('INT8 top score:', float(int8_outputs[0]['scores'][0]))

save_pt2e_int8_artifact(ARTIFACT_PATH, int8_model, metrics={}, extra={'source': 'colab_smoke'})
print('Saved artifact:', ARTIFACT_PATH, 'size MB =', ARTIFACT_PATH.stat().st_size / (1024 * 1024))

reloaded_model, payload = load_pt2e_int8_artifact(str(ARTIFACT_PATH), config, compile_region=False)
reloaded_model = reloaded_model.cpu().eval()
with torch.inference_mode():
    reloaded_outputs = reloaded_model([img.cpu() for img in images])
print('Reloaded keys:', reloaded_outputs[0].keys())
print('Reloaded boxes:', len(reloaded_outputs[0]['boxes']))
if len(reloaded_outputs[0]['scores']) > 0:
    print('Reloaded top score:', float(reloaded_outputs[0]['scores'][0]))
print('Artifact metadata:', payload.get('extra', {}))
print('PT2E convert/save/load smoke test passed')